# ansel-denoise — training (local / Colab / Kaggle, one notebook)

Trains the CFA-domain denoising U-Net on the shard cache published on the
[`shards-v1` release](https://github.com/aurelienpierreeng/ansel-denoise/releases/tag/shards-v1),
with noise synthesized on the fly from the darktable/Ansel noise profiles.
The environment is detected automatically; every cell branches internally,
so the workflow is identical everywhere: **run all cells; after any
disconnect, run all cells again** — training resumes the same fixed cosine
schedule from the newest checkpoint until `TOTAL_STEPS` is reached.

| environment | setup | checkpoint persistence |
|---|---|---|
| **local** | run Jupyter anywhere inside a clone (deps: `pip install -e .[train]`) | `runs/` in the repo |
| **Colab** | `Runtime → Change runtime type → T4 GPU` | Google Drive (`USE_DRIVE`, asks consent) |
| **Kaggle** | Settings: *Accelerator* → GPU, *Internet* → On; optional secrets `KAGGLE_USERNAME`/`KAGGLE_KEY` (Settings → API → Create New Token) | private Kaggle Dataset via API secrets + the committed version output of `/kaggle/working` |

Kaggle only: attach the checkpoint dataset (*Add Input*) after the first
push so later sessions resume from it; publishing the shards once as a
Kaggle Dataset and attaching it skips the multi-GB download at startup.

To **start over cold** (new dataset snapshot, changed hyperparameters), pick
a new `RUN_NAME` — each run name is its own checkpoint directory.

In [ ]:
# --- parameters -----------------------------------------------------------
TOTAL_STEPS = 100_000    # full run length; sessions resume until this is reached
BATCH = 32
PATCH = 128
RUN_NAME = "v1-100k"     # change to start a fresh run (cold restart)
USE_DRIVE = True         # Colab only: persist checkpoints to Google Drive
CKPT_DATASET = f"ansel-denoise-{RUN_NAME}"  # Kaggle only: per-run checkpoint dataset
TIME_BUDGET_H = 'auto'   # wall-clock stop; 'auto' = 8h on Kaggle (hard session cap),
                         # unlimited elsewhere. Set a number to override anywhere.

In [ ]:
# --- environment detection ------------------------------------------------
import importlib.util, os, shutil, subprocess

if os.path.isdir('/kaggle'):
    ENV = 'kaggle'
elif importlib.util.find_spec('google.colab'):
    ENV = 'colab'
else:
    ENV = 'local'
if TIME_BUDGET_H == 'auto':
    TIME_BUDGET_H = 8.0 if ENV == 'kaggle' else None
print(f'environment: {ENV} | time budget: {TIME_BUDGET_H or "unlimited"} h')

if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi', '-L'])
else:
    print('NO CUDA GPU detected'
          + {'colab': ': Runtime -> Change runtime type -> T4 GPU',
             'kaggle': ': Settings -> Accelerator -> GPU',
             'local': ' — training will run on CPU (smoke tests only)'}[ENV])

In [ ]:
# --- code -----------------------------------------------------------------
# local: use the checkout this notebook lives in (found by walking up to
# pyproject.toml) and DO NOT git-pull — the working tree is the user's.
# Colab/Kaggle: clone to an absolute path, pull (not skip) when it already
# exists so a reused runtime always executes the CURRENT code. In all cases
# already-imported modules are purged before re-import.
import os, sys
from pathlib import Path

REPO = None
if ENV == 'local':
    p = Path.cwd()
    while p != p.parent:
        if (p / 'pyproject.toml').is_file():
            REPO = str(p)
            break
        p = p.parent
if REPO is None:
    ROOT = {'kaggle': '/kaggle/tmp', 'colab': '/content', 'local': os.path.expanduser('~')}[ENV]
    os.makedirs(ROOT, exist_ok=True)
    REPO = os.path.join(ROOT, 'ansel-denoise')
    if os.path.isdir(REPO):
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/aurelienpierreeng/ansel-denoise.git', REPO], check=True)
subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'], check=False)
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, 'src'))
for m in [m for m in list(sys.modules) if m.startswith('ansel_denoise')]:
    del sys.modules[m]
import ansel_denoise
print('repo:', REPO, '| package importable:', ansel_denoise.__file__)

In [ ]:
# --- data -----------------------------------------------------------------
# Priority: an existing local shard directory (any 'shards/*' with .npz in
# the repo) > a Kaggle-attached dataset > fetch of the public release into
# an environment-appropriate scratch dir (no auth, public assets).
import json, tarfile, urllib.request
from pathlib import Path

local_dirs = sorted((d for d in Path('shards').glob('*') if any(d.glob('*.npz'))),
                    key=lambda d: -len(list(d.glob('*.npz')))) \
    if Path('shards').is_dir() else []
attached = sorted(Path('/kaggle/input').glob('*/**/*.npz')) if ENV == 'kaggle' else []

if local_dirs:
    SHARDS = local_dirs[0]
    print(f'using local shards: {SHARDS} ({len(list(SHARDS.glob("*.npz")))} shards)')
elif attached:
    SHARDS = attached[0].parent
    print(f'using attached shard dataset: {SHARDS} ({len(list(SHARDS.glob("*.npz")))} shards)')
else:
    SHARDS = Path({'kaggle': '/kaggle/tmp/shards', 'colab': 'shards/rpu',
                   'local': 'shards/rpu'}[ENV])
    SHARDS.mkdir(parents=True, exist_ok=True)
    api = 'https://api.github.com/repos/aurelienpierreeng/ansel-denoise/releases/tags/shards-v1'
    with urllib.request.urlopen(api) as r:
        assets = json.load(r)['assets']
    for a in assets:
        if not a['name'].startswith('shards-'):
            continue
        print(a['name'], f"{a['size'] / 1e6:.0f} MB")
        tmp, _ = urllib.request.urlretrieve(a['browser_download_url'])
        with tarfile.open(tmp) as tar:
            tar.extractall(SHARDS, filter='data')
        Path(tmp).unlink()
    print(f"{len(list(SHARDS.glob('*.npz')))} shards ready")

In [ ]:
# --- checkpoints + resume -------------------------------------------------
# Checkpoint root per environment; resume from the highest NUMBERED
# checkpoint across every persistence layer (never ckpt-final.pt: a stable
# alias that can be staler than a mid-run numbered checkpoint).
from pathlib import Path

out_root = Path('runs')
if ENV == 'colab' and USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        out_root = Path('/content/drive/MyDrive/ansel-denoise-runs')
    except Exception as e:
        print(f'Drive not mounted ({e}); checkpoints stay in the session')
elif ENV == 'kaggle':
    out_root = Path('/kaggle/working/runs')
OUT = out_root / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)

candidates = list(OUT.glob('ckpt-0*.pt'))
if ENV == 'kaggle' and Path('/kaggle/input').exists():
    # pushed checkpoint datasets are flat; committed outputs keep runs/RUN_NAME/
    candidates += list(Path('/kaggle/input').glob('*/ckpt-0*.pt'))
    candidates += list(Path('/kaggle/input').glob(f'*/runs/{RUN_NAME}/ckpt-0*.pt'))
candidates.sort(key=lambda p: int(p.stem.split('-')[1]))
start = int(candidates[-1].stem.split('-')[1]) if candidates else 0
RESUME = ['--resume', str(candidates[-1])] if candidates else []
print(f'checkpoints -> {OUT} | resume from step {start} of {TOTAL_STEPS}'
      + (' — run complete, cells below just re-export' if start >= TOTAL_STEPS else ''))

In [ ]:
# --- train ----------------------------------------------------------------
# One fixed cosine schedule over TOTAL_STEPS; resumed sessions fast-forward
# to the right point of the SAME curve. Runs as a subprocess so a wall-clock
# budget (Kaggle: sessions are killed hard at the cap) stops training in
# time for the persistence/export cells — a timeout is a planned resume, not
# a failure. No --patience: a cosine run is meant to finish its anneal; the
# hard gates (non-finite loss, lr stuck at 0) still abort a broken run.
import os, subprocess, sys

env = dict(os.environ, PYTHONPATH=os.path.join(REPO, 'src'))
cmd = [sys.executable, '-m', 'ansel_denoise.train',
       '--shards', str(SHARDS), '--out', str(OUT),
       '--steps', str(TOTAL_STEPS), '--batch', str(BATCH), '--patch', str(PATCH),
       '--workers', '2', '--val-every', '500', '--ckpt-every', '500',
       '--schedule', 'cosine', *RESUME]
print(' '.join(cmd))
try:
    subprocess.run(cmd, env=env, cwd=REPO, check=True,
                   timeout=TIME_BUDGET_H * 3600 if TIME_BUDGET_H else None)
    print('training reached TOTAL_STEPS')
except subprocess.TimeoutExpired:
    print(f'time budget ({TIME_BUDGET_H} h) reached — checkpoints are saved, '
          f'run all cells again to resume')

In [ ]:
# --- persist checkpoints (Kaggle only) ------------------------------------
# Pushes to a private per-run Kaggle Dataset using the KAGGLE_USERNAME /
# KAGGLE_KEY secrets; skipped gracefully without them (the committed version
# output still carries /kaggle/working). Files are staged FLAT: the kaggle
# CLI's default dir-mode silently skips subdirectories. Colab persists via
# Drive, local via the filesystem — nothing to do for either.
import json, shutil, subprocess
from pathlib import Path

user = None
if ENV == 'kaggle':
    try:
        from kaggle_secrets import UserSecretsClient
        s = UserSecretsClient()
        user, key = s.get_secret('KAGGLE_USERNAME'), s.get_secret('KAGGLE_KEY')
    except Exception as e:
        print(f'no Kaggle API secrets ({e}) — relying on the committed version output only')

if user:
    Path('/root/.kaggle').mkdir(exist_ok=True)
    Path('/root/.kaggle/kaggle.json').write_text(json.dumps({'username': user, 'key': key}))
    Path('/root/.kaggle/kaggle.json').chmod(0o600)

    stage = Path('/kaggle/tmp/ckpt-push')
    shutil.rmtree(stage, ignore_errors=True)
    stage.mkdir(parents=True)
    numbered = sorted(OUT.glob('ckpt-0*.pt'))
    for f in ([numbered[-1]] if numbered else []) + \
             [OUT / 'ckpt-best.pt', OUT / 'ckpt-final.pt', OUT / 'train.log']:
        if f.exists():
            shutil.copy2(f, stage / f.name)
    (stage / 'dataset-metadata.json').write_text(json.dumps({
        'title': CKPT_DATASET, 'id': f'{user}/{CKPT_DATASET}',
        'licenses': [{'name': 'CC0-1.0'}]}))
    step = numbered[-1].stem.split('-')[1] if numbered else '0'
    r = subprocess.run(['kaggle', 'datasets', 'version', '-p', str(stage),
                        '-m', f'{RUN_NAME} step {step}', '-q'], capture_output=True, text=True)
    if r.returncode != 0:  # first push: the dataset does not exist yet
        r = subprocess.run(['kaggle', 'datasets', 'create', '-p', str(stage), '-q'],
                           capture_output=True, text=True)
    print(r.stdout or r.stderr)
    print(f'-> attach kaggle.com/datasets/{user}/{CKPT_DATASET} via Add Input '
          f'so the next session can resume from it')

In [ ]:
# --- export the weights for Ansel -----------------------------------------
from pathlib import Path
from ansel_denoise.export import main as export_main

ckpt = OUT / 'ckpt-final.pt'
if not ckpt.exists():
    print('run not finished yet — no ckpt-final.pt; run all cells again to resume')
else:
    dest = {'kaggle': '/kaggle/working/final.anselnn'}.get(ENV, str(OUT / 'final.anselnn'))
    export_main([str(ckpt), '--out', dest])
    if ENV == 'colab':
        from google.colab import files
        files.download(dest)
    else:
        print('weights:', dest, '(Kaggle: Output tab)' if ENV == 'kaggle' else '')